# DeepSeek-V4-Flash 在 AMD ROCm DSW 上的 vLLM 兼容部署与长上下文验证

这是一份面向 ModelScope Gallery 的代码实践 Notebook。它把一次 DeepSeek-V4-Flash AMD ROCm 部署研究整理成可自动运行的实验复盘：默认 Cell 不下载大模型、不启动重型服务，而是基于已沉淀的实验数据完成环境检查、结果复现、图表生成和优化方向分析。

完整部署和 32K 长上下文重测属于重型实验，已放在 Notebook 后半部分的可选步骤中。

## 运行方式

默认运行目标是“全部 Cell 执行通过，无报错”。因此本 Notebook 分成两类步骤：

- **默认步骤**：环境检查、数据读取、轻量 microbenchmark、图表复现、结论生成。
- **可选步骤**：连接已启动的 vLLM OpenAI API 服务，或按命令重新部署 DeepSeek-V4-Flash。

如果只做 Gallery 自动审核，直接从头运行即可。如果要连接真实服务，请先在终端启动 vLLM，然后设置 `RUN_LIVE_VLLM=1`。

In [ ]:
# 基础依赖：这些包只用于 Notebook 数据分析和可视化。
# Basic dependencies for local analysis and visualization.
from __future__ import annotations

import json
import os
import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import Image, display
except Exception:
    def display(x):
        print(x)
    Image = None

ROOT = Path(os.environ.get("GALLERY_ROOT", ".")).resolve()
if not (ROOT / "data").exists():
    raise FileNotFoundError(
        f"未找到 data/ 目录，请在 Gallery 发布包根目录运行 Notebook，当前目录：{ROOT}"
    )

DATA_DIR = ROOT / "data"
ASSET_DIR = ROOT / "assets"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Gallery root:", ROOT)
print("Data files:", sorted(p.name for p in DATA_DIR.glob("*.csv")))

In [ ]:
# 环境检查：记录 Python、系统、PyTorch/ROCm 信息。
# Environment check: this cell never fails when torch/ROCm is unavailable.
env_report = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cwd": str(Path.cwd()),
}

try:
    import torch
    env_report["torch"] = getattr(torch, "__version__", "unknown")
    env_report["torch_hip"] = getattr(getattr(torch, "version", None), "hip", None)
    env_report["cuda_available_api"] = bool(torch.cuda.is_available())
    env_report["device_count"] = int(torch.cuda.device_count()) if torch.cuda.is_available() else 0
    env_report["device_name_0"] = (
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
    )
except Exception as exc:
    env_report["torch"] = "not_available"
    env_report["torch_error"] = f"{type(exc).__name__}: {exc}"

print(json.dumps(env_report, indent=2, ensure_ascii=False))
(OUTPUT_DIR / "env_report.json").write_text(
    json.dumps(env_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

In [ ]:
# 轻量 top-k microbenchmark。
# 在 ROCm DSW 中，PyTorch 的 torch.cuda API 对应 HIP/ROCm 后端；
# 如果当前环境没有 ROCm GPU，则自动退回 NumPy CPU 路径，保证 Gallery 自动运行不失败。
def benchmark_topk(size: int = 200_000, k: int = 4096) -> dict:
    try:
        import torch
        if torch.cuda.is_available():
            device = "cuda"
            x = torch.randn(size, device=device)
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            vals, idx = torch.topk(x, k)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - t0
            return {
                "backend": "torch_rocm_cuda_api",
                "size": size,
                "k": k,
                "elapsed_ms": elapsed * 1000,
                "top_value": float(vals[0].detach().cpu()),
            }
    except Exception as exc:
        torch_error = f"{type(exc).__name__}: {exc}"
    else:
        torch_error = "torch unavailable or no ROCm GPU"

    rng = np.random.default_rng(20260524)
    x = rng.standard_normal(size)
    t0 = time.perf_counter()
    idx = np.argpartition(x, -k)[-k:]
    vals = np.sort(x[idx])[::-1]
    elapsed = time.perf_counter() - t0
    return {
        "backend": "numpy_cpu_fallback",
        "size": size,
        "k": k,
        "elapsed_ms": elapsed * 1000,
        "top_value": float(vals[0]),
        "note": torch_error,
    }

microbench = benchmark_topk()
print(json.dumps(microbench, indent=2, ensure_ascii=False))
(OUTPUT_DIR / "topk_microbenchmark.json").write_text(
    json.dumps(microbench, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

In [ ]:
# 读取本次 DeepSeek-V4-Flash ROCm 实验数据。
topk_df = pd.read_csv(DATA_DIR / "8k_topk_sweep.csv")
correct_df = pd.read_csv(DATA_DIR / "context_correctness.csv")
service_df = pd.read_csv(DATA_DIR / "service_checks.csv")
patch_df = pd.read_csv(DATA_DIR / "compat_patch_summary.csv")
negative_df = pd.read_csv(DATA_DIR / "negative_cases.csv")

print("8K topk sweep:")
display(topk_df)

print("Context correctness:")
display(correct_df)

print("Service checks:")
display(service_df)

In [ ]:
# 基础一致性断言：这些断言对应文章中的关键结论。
# Assertions keep the notebook self-checking and reviewer friendly.
assert set(topk_df["needle_pass"]) == {"PASS"}
assert int(topk_df.loc[topk_df["ttft_s"].idxmin(), "index_topk"]) == 2048
assert int(topk_df.loc[topk_df["effective_prefill_tok_s"].idxmax(), "index_topk"]) == 2048

pass_32k = correct_df[
    (correct_df["context_tokens"] == 32768)
    & (correct_df["needle_position"] == "begin")
    & (correct_df["passed"] == True)
]
assert not pass_32k.empty
assert int(pass_32k.iloc[0]["index_topk"]) == 4096

assert service_df["passed"].all()

summary = {
    "best_8k_topk_by_ttft": int(topk_df.loc[topk_df["ttft_s"].idxmin(), "index_topk"]),
    "best_8k_ttft_s": float(topk_df["ttft_s"].min()),
    "best_8k_prefill_tok_s": float(topk_df["effective_prefill_tok_s"].max()),
    "first_verified_32k_begin_topk": int(pass_32k.iloc[0]["index_topk"]),
    "service_checks_passed": int(service_df["passed"].sum()),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
(OUTPUT_DIR / "result_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

In [ ]:
# 画图 1：8K topk sweep 中 TTFT 与等效 prefill 的变化。
fig, ax1 = plt.subplots(figsize=(8, 4.6), dpi=140)
x = topk_df["index_topk"].astype(str)

ax1.bar(x, topk_df["ttft_s"], color="#635bff", alpha=0.82, label="TTFT (s)")
ax1.set_xlabel("index_topk")
ax1.set_ylabel("TTFT / first token latency (s)")
ax1.grid(axis="y", linestyle="--", alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(x, topk_df["effective_prefill_tok_s"], color="#00a88f", marker="o", linewidth=2.5, label="Prefill tok/s")
ax2.set_ylabel("Effective prefill (tok/s)")

best = int(topk_df.loc[topk_df["ttft_s"].idxmin(), "index_topk"])
fig.suptitle(f"8K topk sweep: best TTFT at index_topk={best}")
fig.tight_layout()

out_path = OUTPUT_DIR / "8k_topk_sweep.png"
fig.savefig(out_path, bbox_inches="tight")
plt.close(fig)
print("Saved:", out_path)
if Image is not None:
    display(Image(filename=str(out_path)))

In [ ]:
# 画图 2：不同 topk 与上下文长度下的语义正确性矩阵。
begin_df = correct_df[correct_df["needle_position"] == "begin"].copy()
pivot = begin_df.pivot_table(
    index="index_topk",
    columns="context_tokens",
    values="passed",
    aggfunc="max",
).fillna(False)
pivot = pivot.sort_index().sort_index(axis=1)

fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=140)
matrix = pivot.astype(int).values
im = ax.imshow(matrix, cmap="YlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{int(c/1024)}K" for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([str(i) for i in pivot.index])
ax.set_xlabel("Context length")
ax.set_ylabel("index_topk")
ax.set_title("Begin-position needle retrieval correctness")

for y in range(matrix.shape[0]):
    for x_i in range(matrix.shape[1]):
        ax.text(x_i, y, "PASS" if matrix[y, x_i] else "FAIL", ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax, ticks=[0, 1], label="pass")
fig.tight_layout()
out_path = OUTPUT_DIR / "context_correctness_matrix.png"
fig.savefig(out_path, bbox_inches="tight")
plt.close(fig)
print("Saved:", out_path)
if Image is not None:
    display(Image(filename=str(out_path)))

In [ ]:
# 展示兼容补丁与负例：这部分对应 ROCm 工程排障经验。
print("Compatibility patch summary:")
display(patch_df)

print("Negative cases:")
display(negative_df)

In [ ]:
# 可选：如果当前机器已经启动了 vLLM OpenAI API server，可以打开 live check。
# 默认不启用，避免自动审核时因为没有大模型服务而失败。
RUN_LIVE_VLLM = os.environ.get("RUN_LIVE_VLLM", "0") == "1"
VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://127.0.0.1:8000")
SERVED_MODEL = os.environ.get("SERVED_MODEL", "deepseek-v4-flash-amd-32k")

live_result = {"enabled": RUN_LIVE_VLLM, "base_url": VLLM_BASE_URL}
if not RUN_LIVE_VLLM:
    live_result["status"] = "skipped"
    live_result["reason"] = "Set RUN_LIVE_VLLM=1 to query a running vLLM service."
else:
    try:
        r = requests.get(f"{VLLM_BASE_URL}/v1/models", timeout=10)
        live_result["models_status_code"] = r.status_code
        live_result["models_body_head"] = r.text[:500]

        payload = {
            "model": SERVED_MODEL,
            "messages": [
                {"role": "user", "content": "What is the capital of France? Answer with one word."}
            ],
            "max_tokens": 8,
            "temperature": 0,
        }
        r2 = requests.post(
            f"{VLLM_BASE_URL}/v1/chat/completions",
            headers={"Content-Type": "application/json"},
            data=json.dumps(payload),
            timeout=60,
        )
        live_result["chat_status_code"] = r2.status_code
        live_result["chat_body_head"] = r2.text[:500]
    except Exception as exc:
        live_result["status"] = "error"
        live_result["error"] = f"{type(exc).__name__}: {exc}"

print(json.dumps(live_result, indent=2, ensure_ascii=False))
(OUTPUT_DIR / "optional_live_vllm_check.json").write_text(
    json.dumps(live_result, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

## 可选完整部署命令

下面这部分是重型实验说明，不作为默认自动运行 Cell。原因是 DeepSeek-V4-Flash 权重体积很大，完整部署还涉及 vLLM、ROCm/AITER、补丁顺序和长上下文测试，直接放进默认 Cell 容易导致 Gallery 自动审核超时。

推荐做法：

1. 默认 Notebook 复现实验数据和图表；
2. 在 AMD ROCm DSW 终端里按脚本恢复补丁和服务；
3. 服务启动后，再把 `RUN_LIVE_VLLM=1` 打开，运行上面的 live check Cell。

In [ ]:
# 生成一份可选重实验命令文件，供在 DSW 终端中手动执行。
commands = r'''#!/usr/bin/env bash
set -euo pipefail

# 这些路径请按自己的 DSW 实例调整。
export REPRO_DIR=${REPRO_DIR:-$(pwd)}
export MODEL_DIR=${MODEL_DIR:-/path/to/DeepSeek-V4-Flash}
export RUN_MODEL_DIR=${RUN_MODEL_DIR:-/home/deepseek_hybrid/DeepSeek-V4-Flash}
export SERVED_MODEL_NAME=${SERVED_MODEL_NAME:-deepseek-v4-flash-amd-32k-batch8-16384}

# 1. 只读检查环境
python3 - <<'PY'
import torch
print("torch:", torch.__version__)
print("hip:", getattr(torch.version, "hip", None))
print("cuda_available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
PY

# 2. 恢复补丁。具体脚本来自本次研究沉淀的 repro bundle。
bash "$REPRO_DIR/scripts/restore_current_patched_site_packages_any_site.sh"
bash "$REPRO_DIR/scripts/patch_home_model_config_index_topk.sh" 4096

# 3. 启动 32K correctness-first baseline。
nohup bash "$REPRO_DIR/scripts/launch_32k_batch8_16384_system.sh" \
  > /home/amd_research_logs/gallery_launch_32k.log 2>&1 &

# 4. 服务就绪后运行健康检查和 needle gate。
python3 "$REPRO_DIR/scripts/run_openai_gate.py" \
  --model "$SERVED_MODEL_NAME" \
  --name capital_chat \
  --timeout 120

python3 "$REPRO_DIR/scripts/run_openai_gate.py" \
  --model "$SERVED_MODEL_NAME" \
  --name needle8192_tokens \
  --timeout 1200
'''

cmd_path = OUTPUT_DIR / "optional_heavy_reproduce_commands.sh"
cmd_path.write_text(commands, encoding="utf-8")
print("Wrote optional command file:", cmd_path)
print(commands)

In [ ]:
# 最终汇总：生成 Gallery 运行摘要，方便审核者查看。
gallery_summary = {
    "topic": "DeepSeek-V4-Flash AMD ROCm vLLM deployment baseline",
    "default_cells_require_model_weights": False,
    "default_cells_start_vllm": False,
    "amd_rocm_related": True,
    "data_files": sorted(p.name for p in DATA_DIR.glob("*.csv")),
    "generated_outputs": sorted(p.name for p in OUTPUT_DIR.glob("*")),
    "key_results": summary,
}

summary_path = OUTPUT_DIR / "gallery_run_summary.json"
summary_path.write_text(json.dumps(gallery_summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(gallery_summary, indent=2, ensure_ascii=False))
print("Gallery notebook default run finished successfully.")

## 小结

这份 Notebook 把 DeepSeek-V4-Flash 在 AMD ROCm DSW 上的部署研究拆成了三层：

1. **可自动运行层**：环境检查、轻量 top-k microbenchmark、CSV 数据分析和图表生成；
2. **可验证结果层**：短问答、2K/8K/32K needle retrieval、8K topk sweep、负例记录；
3. **可继续研究层**：ROCm-native sparse MLA、top-k、mHC、MoE、FlashMLA 和投机解码。

默认运行不依赖大模型权重，因此更适合 Gallery 自动审核；完整部署仍然可以通过可选脚本在 AMD GPU DSW 实例中复现实验。